# Partie 4 : Visualisation en temps reel

Ce notebook propose une UI interactive pour afficher les donnees InfluxDB en direct.

Fonctionnalites :
- selection du champ (`temp`, `hum`, `co2`, `power`, `occupied`)
- selection de la fenetre temporelle
- reglage de la frequence de rafraichissement
- boutons `Demarrer` / `Arreter`

Prerequis : lancer d'abord `notebooks/0_db-filler.ipynb` pour generer des donnees en continu.

In [ ]:
from influxdb_client import InfluxDBClient
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets
from threading import Event, Thread, Lock

# Config InfluxDB (identique au script de generation)
URL = "http://influxdb2:8086"
TOKEN = "MyInitialAdminToken0=="
ORG = "docs"
BUCKET = "home"
MEASUREMENT = "home"

client = InfluxDBClient(url=URL, token=TOKEN, org=ORG)
query_api = client.query_api()

_worker_stop_event = None
_worker_thread = None
_render_lock = Lock()


def fetch_last_window(field: str = "temp", window: str = "15m") -> pd.DataFrame:
    flux_query = f'''
from(bucket: "{BUCKET}")
  |> range(start: -{window})
  |> filter(fn: (r) => r._measurement == "{MEASUREMENT}")
  |> filter(fn: (r) => r._field == "{field}")
  |> keep(columns: ["_time", "_value", "room"])
  |> sort(columns: ["_time"])
'''

    tables = query_api.query(flux_query)

    rows = []
    for table in tables:
        for record in table.records:
            rows.append(
                {
                    "time": record.get_time(),
                    "room": record.values.get("room", "unknown"),
                    "value": record.get_value(),
                }
            )

    if not rows:
        return pd.DataFrame(columns=["time", "room", "value"])

    df = pd.DataFrame(rows)
    df["time"] = pd.to_datetime(df["time"], utc=True)
    return df


def render_plot_once(field: str, window: str, output: widgets.Output) -> None:
    df = fetch_last_window(field=field, window=window)

    with _render_lock:
        with output:
            output.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(11, 4))

            if df.empty:
                ax.text(0.5, 0.5, "Aucune donnee disponible", ha="center", va="center")
                ax.set_axis_off()
            else:
                for room, group in df.groupby("room"):
                    ax.plot(group["time"], group["value"], label=room)

                ax.legend(loc="upper left", ncol=3)
                ax.set_title(f"{field} - fenetre glissante {window}")
                ax.set_xlabel("Temps")
                ax.set_ylabel(field)
                ax.grid(alpha=0.25)
                fig.autofmt_xdate()

            display(fig)
            plt.close(fig)


def _live_worker(field_widget, window_widget, refresh_widget, output: widgets.Output, stop_event: Event) -> None:
    while not stop_event.is_set():
        render_plot_once(field_widget.value, window_widget.value, output)
        stop_event.wait(float(refresh_widget.value))


def _stop_worker() -> None:
    global _worker_stop_event, _worker_thread

    if _worker_stop_event is not None:
        _worker_stop_event.set()

    if _worker_thread is not None and _worker_thread.is_alive():
        _worker_thread.join(timeout=1.0)

    _worker_stop_event = None
    _worker_thread = None


def launch_dashboard_ui() -> None:
    global _worker_stop_event, _worker_thread

    # Stoppe une ancienne boucle live si elle existe deja
    _stop_worker()

    field_widget = widgets.Dropdown(
        options=["temp", "hum", "co2", "power", "occupied"],
        value="temp",
        description="Champ",
        layout=widgets.Layout(width="260px"),
    )

    window_widget = widgets.Dropdown(
        options=["5m", "15m", "30m", "1h", "6h", "24h"],
        value="15m",
        description="Fenetre",
        layout=widgets.Layout(width="260px"),
    )

    refresh_widget = widgets.FloatSlider(
        value=2.0,
        min=0.5,
        max=10.0,
        step=0.5,
        description="Refresh (s)",
        continuous_update=False,
        readout_format=".1f",
        layout=widgets.Layout(width="360px"),
    )

    start_button = widgets.Button(description="Demarrer", button_style="success", icon="play")
    stop_button = widgets.Button(description="Arreter", button_style="warning", icon="stop")
    refresh_once_button = widgets.Button(description="Rafraichir", icon="refresh")
    status = widgets.HTML("<b>Statut:</b> pret")
    output = widgets.Output()

    def on_start_clicked(_):
        global _worker_stop_event, _worker_thread

        if _worker_thread is not None and _worker_thread.is_alive():
            status.value = "<b>Statut:</b> deja en cours"
            return

        _worker_stop_event = Event()
        _worker_thread = Thread(
            target=_live_worker,
            args=(field_widget, window_widget, refresh_widget, output, _worker_stop_event),
            daemon=True,
        )
        _worker_thread.start()
        status.value = "<b>Statut:</b> en cours"

    def on_stop_clicked(_):
        _stop_worker()
        status.value = "<b>Statut:</b> arrete"

    def on_refresh_once_clicked(_):
        render_plot_once(field_widget.value, window_widget.value, output)
        status.value = "<b>Statut:</b> apercu mis a jour"

    def on_selector_change(_):
        # Si non demarre, on met a jour l'aperçu immediatement pour valider la selection
        if _worker_thread is None or not _worker_thread.is_alive():
            render_plot_once(field_widget.value, window_widget.value, output)
        status.value = f"<b>Statut:</b> champ={field_widget.value}, fenetre={window_widget.value}"

    start_button.on_click(on_start_clicked)
    stop_button.on_click(on_stop_clicked)
    refresh_once_button.on_click(on_refresh_once_clicked)

    field_widget.observe(on_selector_change, names="value")
    window_widget.observe(on_selector_change, names="value")

    controls = widgets.HBox([field_widget, window_widget, refresh_widget])
    actions = widgets.HBox([start_button, stop_button, refresh_once_button, status])
    display(widgets.VBox([controls, actions, output]))

    # Affiche un premier rendu immediat
    render_plot_once(field_widget.value, window_widget.value, output)


def close_influx_client() -> None:
    _stop_worker()
    client.close()

In [ ]:
# Lance l'interface de visualisation
# 1) Choisir champ/fenetre/refresh
# 2) Cliquer sur Rafraichir pour tester la selection
# 3) Cliquer sur Demarrer pour le live
# 4) Cliquer sur Arreter pour stopper la boucle
launch_dashboard_ui()

In [ ]:
#fermer proprement le client InfluxDB
close_influx_client()